# 🏗️ Simbolička matematika u Pythonu — SymPy za građevinare
### Računalno programiranje u građevinarstvu (RPuG) — GFOS Osijek
---
**Biblioteka:** `sympy`  
**Okruženje:** Google Colab (automatski LaTeX render)  
**Tematika:** Beton · Čelik · Drvo · Eurokod

> **Napomena za studente:** Varijable u svakom primjeru su unaprijed zadane.  
> Ostatak koda **pišemo zajedno** — prati upute i komentare `# TODO:`.

---
## 0. Postavljanje okoline

Prije svega potrebno je uvesti (`import`) biblioteku **SymPy** i aktivirati  
automatski lijepi ispis koji u Google Colabu renderira simboličke izraze kao  
prave matematičke formule (LaTeX).

| Naredba | Opis |
|---|---|
| `import sympy as sp` | uvoz cijele biblioteke pod pseudonimom `sp` |
| `from sympy import ...` | uvoz točno određenih funkcija |
| `sp.init_printing()` | aktivira automatski LaTeX render u notebooku |

In [ ]:
import sympy as sp
from sympy import (
    symbols, Symbol, Rational, sqrt, pi, oo, I,
    sin, cos, tan, exp, log,
    diff, integrate, limit, series,
    solve, Eq, linsolve,
    Matrix, latex, simplify, expand, factor,
    collect, cancel, apart, together,
    trigsimp, pprint
)
from IPython.display import display, Math

# Aktiviramo automatski LaTeX render u Colabu
sp.init_printing()

print("SymPy verzija:", sp.__version__)
print("Postavljanje okoline: OK ✓")

---
## 1. Simboličke varijable i izrazi

### Zašto simbolička matematika?

Standardni Python radi s **brojevima** (float, int).  
SymPy radi s **matematičkim simbolima** — može točno izraziti `1/3`,  
`√2`, `π`, derivacije, integrale i algebarske jednadžbe.

### Deklaracija simbola

```python
x = symbols('x')                          # realna varijabla
a, b = symbols('a b', real=True)          # više simbola odjednom, realni
n   = symbols('n', integer=True)          # cijeli broj
L   = symbols('L', positive=True)         # pozitivna veličina
```

> **Važno:** Bez deklaracije (`symbols(...)`), Python ne zna da je `x` varijabla —  
> tretira je kao nedefinirani naziv i javlja grešku!

### Razlika: Python float vs SymPy Rational

```
0.333...  →  neprecizni float
Rational(1,3)  →  točna simbolička vrijednost 1/3
```

---
## 2. Algebarska manipulacija izrazima — Čelični presjek (EC3)

### Ključne funkcije

| Naredba | Opis | Primjer |
|---|---|---|
| `expand(izraz)` | Razvija zagrade | `expand((a+b)**2)` → `a²+2ab+b²` |
| `factor(izraz)` | Faktorizira | `factor(a**2-b**2)` → `(a-b)(a+b)` |
| `simplify(izraz)` | Opća simplifikacija | uklanja redundantne operacije |
| `collect(izraz, x)` | Grupira po varijabli | korisno za polinome |
| `cancel(izraz)` | Kraćenje razlomaka | `cancel((x**2-1)/(x-1))` → `x+1` |

### Primjer: Plastični moment otpora dvostruko-simetričnog I-presjeka

Plastični moment otpora definiran je kao:

$$W_{pl} = b_f \cdot t_f \cdot (h - t_f) + \frac{t_w \cdot h_w^2}{4}$$

Uvjet nosivosti na savijanje (EC3-1-1, §6.2.5):

$$M_{Ed} \leq M_{pl,Rd} = \frac{W_{pl} \cdot f_y}{\gamma_{M0}}$$

In [ ]:
from sympy import symbols, expand, factor, collect, Eq

# ── Zadane varijable — I-presjek (čelik) ────────────────────────────────
b_f, t_f  = symbols('b_f t_f', positive=True)   # širina i debljina pojasnice [m]
t_w, h_w  = symbols('t_w h_w', positive=True)   # debljina i visina hrpta [m]
h         = symbols('h', positive=True)         # ukupna visina presjeka [m]
f_y       = symbols('f_y', positive=True)       # granica tečenja čelika [MPa]
gamma_M0  = symbols('gamma_M0', positive=True)  # faktor pouzdanosti materijala [-]
M_Ed      = symbols('M_Ed', positive=True)      # projektna vrijednost momenta [kNm]

# ── TODO: Plastični moment otpora ────────────────────────────────────────
# Wpl = b_f * t_f * (h - t_f) + t_w * h_w**2 / 4
W_pl = ...  # TODO: upiši izraz

# Plastična momentna nosivost Mpl,Rd = W_pl * f_y / gamma_M0
M_plRd = ...  # TODO: upiši izraz

# ── Razvijanje i pojednostavljivanje ─────────────────────────────────────
W_pl_razvijen = expand(W_pl)   # razvijanje zagrada

# Faktoriziran M_plRd
M_plRd_factor = factor(M_plRd)   # faktorizacija (zajednički faktori)

# Grupiranje M_plRd
M_plRd_collect = collect(M_plRd_factor, b_f)   # grupiranje po širini pojasnice

print("Plastični moment otpora W_pl:")
display(Eq(symbols('W_pl'), W_pl))

print("W_pl (prošireno, razvijeno):")
display(Eq(symbols('W_pl'), W_pl_razvijen))

print("Plastična momentna nosivost M_pl,Rd:")
display(Eq(symbols('M_plRd'), M_plRd))

print("Faktorizirana plastična momentna nosivost, M_pl,Rd:")
display(Eq(symbols('M_plRd'), M_plRd_factor))

print("Grupirana plastična momentna nosivost, M_pl,Rd:")
display(Eq(symbols('M_plRd'), M_plRd_collect))

In [ ]:
# ── Numerička provjera za HEA 200 (S235) ────────────────────────────────
# Dimenzije HEA 200: bf=200mm, tf=10mm, hw=160mm, tw=6.5mm
# Materijal S235: fy=235 MPa, gamma_M0=1.00

# Numerička provjera – HEA 200, S235
vrijednosti = {
    h: 0.190,       # m – ukupna visina
    b_f: 0.200,     # m – širina pojasnice
    t_f: 0.010,     # m – debljina pojasnice
    h_w: 0.170,     # m – visina hrpta (= h - 2·t_f = 0.190 - 0.020)
    t_w: 0.0065,    # m – debljina hrpta
    f_y: 235e3,     # kN/m² (= 235 MPa)
    gamma_M0: 1.00
}

# ── TODO: Supstitucija i numerički rezultat ───────────────────────────────
# Koristimo metodu .subs(rjecnik) za uvrštavanje vrijednosti
W_pl_num   = ...   # TODO: W_pl.subs(vrijednosti)
M_plRd_num = ...  # TODO: M_plRd.subs(vrijednosti)

# 1e6 pretvara m³ → cm³ (1 m³ = 1.000.000 cm³)
print(f"W_pl  (HEA 200) = {float(W_pl_num)*1e6:.1f} cm³")
print(f"M_pl,Rd (HEA 200, S235) = {float(M_plRd_num):.2f} kNm")

---
## 3. Jednadžbe i sustavi — Reakcije oslonaca nosača

### Ključne funkcije

| Naredba | Opis |
|---|---|
| `Eq(lijevo, desno)` | Definira jednadžbu `lijevo = desno` |
| `solve(jednadžba, nepoznanica)` | Rješava jednadžbu za nepoznanicu |
| `solve([j1, j2], [x, y])` | Rješava **sustav** jednadžbi |
| `linsolve([j1,j2], x, y)` | Linearni sustav (učinkovitije) |

### Primjer: Slobodno oslonjeni nosač s konzolom

```
 q [kN/m]       F [kN]
 ↓↓↓↓↓↓↓↓↓↓    ↓
 ┬──────────────┬──────┐
 A              B      C
 |←────── L ───→|←ˡc──→|

Uvjeti ravnoteže:
  ΣFy = 0:  R_A + R_B = q·L + F
  ΣM_A = 0: R_B·L    = q·L²/2 + F·(L+l_c)
```

In [ ]:
from sympy import symbols, Eq, solve, simplify

# ── Zadane varijable — geometrija i opterećenje ─────────────────────────
L   = symbols('L',   positive=True)   # raspon A-B [m]
lc  = symbols('l_c', positive=True)   # duljina konzole [m]
q   = symbols('q',   positive=True)   # kontinuirano opterećenje [kN/m]
F   = symbols('F',   positive=True)   # koncentrirana sila na konzoli [kN]
R_A = symbols('R_A')                   # reakcija u A [kN]
R_B = symbols('R_B')                   # reakcija u B [kN]

# ── TODO: Jednadžbe ravnoteže ─────────────────────────────────────────────
# Jednadžba 1: suma sila u smjeru y  →  R_A + R_B - q*L - F = 0
# ΣFy = 0
jednadzba_sila = ...   # TODO: Eq(R_A + R_B, q*L + F)

# Jednadžba 2: suma momenata oko A  →  R_B*L - q*L**2/2 - F*(L+lc) = 0
# ΣM_A = 0 (pozitivan moment u smjeru kazaljke na satu)
jednadzba_mom  = ...   # TODO: Eq(R_B*L, q*L**2/2 + F*(L+lc))

# ── TODO: Rješavanje sustava ──────────────────────────────────────────────
# Rješavamo sustav za nepoznanice RA i RB
# solve([jednadzba_sila, jednadzba_mom], [R_A, R_B])  →  vraca rjecnik s rjesenjima
rjesenje = solve([jednadzba_sila, jednadzba_mom], [R_A, R_B])

print("Reakcija u osloncu A:")
display(Eq(RA, rjesenje[R_A]))

print("Reakcija u osloncu B:")
display(Eq(RB, rjesenje[R_B]))

In [ ]:
# ── TODO: Dijagram unutarnjih sila ────────────────────────────────────────
# Udaljenost x od oslonca A (0 ≤ x ≤ L)
# Unutarnje sile – uvrstimo izračunate reakcije
x = symbols('x', nonnegative=True)   # udaljenost od oslonca A [m]

# Uvrstimo simboličke reakcije i zapišemo poprečnu silu V(x) i moment M(x)
# V(x) = R_A - q*x
# M(x) = R_A*x - q*x**2/2
# V(x) i M(x) s uvrštenim R_A (iz rješenja sustava!)
V_x = ... # TODO: upiši izraz za V(x) = rjesenje[R_A] - q*x
M_x = ... # TODO: upiši izraz za M(x) = rjesenje[R_A]*x - q*x**2/2

# Nalazimo položaj gdje je V(x) = 0  →  maksimalni moment
x_max = solve(Eq(V_x, 0), x)[0]
M_max = simplify(M_x.subs(x, x_max))

print("Poprečna sila V(x):")
display(Eq(symbols('V(x)'), V_x))

print("Moment savijanja M(x):")
display(Eq(symbols('M(x)'), M_x))

print("Položaj maksimalnog momenta x_max:")
display(Eq(symbols('x_max'), simplify(x_max)))

print("Maksimalni moment M_max:")
display(Eq(symbols('M_max'), simplify(M_max)))


# ── Numerička provjera ─────────────────────────────────────────────────
vrijednosti = {L: 5.0, lc: 1.5, q: 10.0, F: 20.0}

R_A_num = rjesenje[R_A].subs(vrijednosti)
R_B_num = rjesenje[R_B].subs(vrijednosti)
x_max_num = x_max.subs(vrijednosti)
M_max_num = M_max.subs(vrijednosti)

print(f"\nNumerička provjera (L=5m, lc=1.5m, q=10kN/m, F=20kN):")
print(f"R_A = {R_A_num:.2f} kN")
print(f"R_B = {R_B_num:.2f} kN")
print(f"x_max = {x_max_num:.2f} m")
print(f"M_max = {M_max_num:.2f} kNm")

---
## 4. Derivacija — Elastična linija AB-grede (EC2)

### Ključne naredbe

| Naredba | Opis |
|---|---|
| `diff(f, x)` | Prva derivacija f po x: df/dx |
| `diff(f, x, 2)` | Druga derivacija: d²f/dx² |
| `diff(f, x, n)` | n-ta derivacija |

### Veza između opterećenja, sila i progiba:

$$-q(x) = EI \frac{d^4 w}{dx^4}, \quad V(x) = EI \frac{d^3 w}{dx^3}, \quad M(x) = EI \frac{d^2 w}{dx^2}$$

$$\theta(x) = \frac{dw}{dx}, \quad w(x) = \text{progib}$$

### Primjer: Progib slobodno oslonjene grede pod jednoliko distribuiranih (kontinuiranim) opterećenjem

Elastična linija:
$$w(x) = \frac{q}{24EI}\left(2Lx^3 - x^4 - L^3 x\right)$$

In [ ]:
# ── Zadane varijable ─────────────────────────────────────────────────────
x, L = symbols('x L', positive=True)
q    = symbols('q', positive=True)          # raspodijeljeno opterećenje [kN/m]
E, I = symbols('E I', positive=True)        # modul elastičnosti, moment tromosti

# ── Elastična linija (zadana formula — upiši s nastavnikom) ───────────────
# w(x) = q / (24*E*I) * (2*L*x**3 - x**4 - L**3*x)
w = ...   # TODO: upiši izraz

# ── TODO: Derivacije ──────────────────────────────────────────────────────
# Rotacija (Kut nagiba)
theta = ...   # TODO: diff(w, x)

# Moment savijanja iz elastične linije: M = E*I * d²w/dx²
M = ...   # TODO: E*I * diff(w, x, 2)

# Poprečna sila: V = E*I * d³w/dx³
V = ...   # TODO: E*I * diff(w, x, 3)

# Opterećenje: q_provjera = E*I * d⁴w/dx⁴  (trebalo bi dati q)
q_provjera = E*I * diff(w, x, 4)

print("Elastična linija w(x):")
display(Eq(symbols('w(x)'), w))

print("Rotacija φ(x) = dw/dx:")
display(Eq(symbols('theta(x)'), simplify(theta)))

print("Moment M(x) = EI·d²w/dx²:")
display(Eq(symbols('M(x)'), simplify(M)))

print("Poprečna sila V(x) = EI·d³w/dx³:")
display(Eq(symbols('V(x)'), simplify(V)))

print("Provjera opterećenja EI·d⁴w/dx⁴ = -q?", simplify(q_provjera) == -q)

In [ ]:
# ── TODO: Maksimalni progib u sredini raspona ─────────────────────────────
# Progib je maksimalan u sredini (x = L/2) za simetričan slučaj

w_max = w.subs(x, L/2)
w_max = simplify(w_max)

# Standardna formula: w_max = 5*q*L**4 / (384*E*I)
w_formula = Rational(5, 384) * q * L**4 / (E*I)

print("Maksimalni progib w(L/2):")
display(Eq(symbols('w_max'), w_max))

print("Standardna formula 5qL⁴/384EI:")
display(Eq(symbols('w_max'), w_formula))

print("Jednaki su?", simplify(w_max + w_formula) == 0)

---
## 5. Integrali — Moment tromosti složenog presjeka

### Ključne naredbe

| Naredba | Opis |
|---|---|
| `integrate(f, x)` | Neodređeni integral ∫f dx |
| `integrate(f, (x, a, b))` | Određeni integral ∫ₐᵇ f dx |
| `integrate(f, (x,a,b), (y,c,d))` | Dvostruki integral |

### Steinerov teorem (paralelna os):

$$I_{ukupno} = \sum_i \left( I_{i,c} + A_i \cdot e_i^2 \right)$$

### Primjer: T-presjek (AB greda) — integriranjem

Moment tromosti T-presjeka integracijom po površini:

$$I_y = \int\int z^2 \, dA$$

In [ ]:
from sympy import symbols, integrate, simplify, Eq

# ── Zadane varijable — T-presjek ─────────────────────────────────────────
# Pojasnica (gornja): širina b_f, debljina t_f
# Hrbat:            širina b_w, visina h_w
# Koordinate (z) mjeri se od dna hrpta prema gore
z = symbols('z')                          # koordinata od dna presjeka [m]
bf, tf = symbols('b_f t_f', positive=True)  # pojasnica: širina, debljina
bw, hw = symbols('b_w h_w', positive=True)  # hrbat: širina, visina
h_tot   = hw + tf                                   # ukupna visina presjeka

# Površine komponenti
A_f = bf * tf    # pojasnica (flange)
A_w = bw * hw    # hrbat (web)

# Težišta komponenti – mjereno od dna presjeka
z_f = hw + tf/2  # težište pojasnice
z_w = hw / 2     # težište hrpta

# ── TODO: Položaj ukupnog težišta ─────────────────────────────────────────
# z_c = (A_f * z_f + A_w * z_w) / (A_f + A_w)
# Položaj ukupnog težišta (od dna)
z_c = (A_f * z_f + A_w * z_w) / (A_f + A_w)
z_c = simplify(z_c)

print("Položaj ukupnog težišta od dna:")
display(Eq(symbols('z_c'), z_c))

In [ ]:
# ── TODO: Momenti tromosti integracijom ─────────────────────────────────
# Momenti tromosti integracijom oko vlastitih centralnih osi
# Granice integracije za pojasnicu: z ∈ [hw, hw+tf]
# Granice integracije za hrbat:      z ∈ [0, hw]
# I_f_c = simplify(integrate(bf * (z - z_f)**2, (z, hw, hw + tf)))
# I_w_c = simplify(integrate(bw * (z - z_w)**2, (z, 0, hw)))
I_f_c = ... # TODO: upiši simplify(integrate(bf * (z - z_f)**2, (z, hw, hw + tf)))
I_w_c = ... # TODO: upiši simplify(integrate(bw * (z - z_w)**2, (z, 0, hw)))

# Steinerov doprinos – prijelaz na os kroz ukupno težište
e_f = z_f - z_c   # ekscentricitet pojasnice
e_w = z_w - z_c   # ekscentricitet hrpta

I_Steiner_f = A_f * e_f**2
I_Steiner_w = A_w * e_w**2

# Ukupni moment tromosti oko težišta T-presjeka
I_y = simplify(I_f_c + I_Steiner_f + I_w_c + I_Steiner_w)

print("\nUkupni moment tromosti T-presjeka oko težišta:")
display(Eq(symbols('I_y'), I_y))

# Numerička provjera
vrijednosti = {
    bf: 0.40,   # m
    tf: 0.10,   # m
    bw: 0.15,   # m
    hw: 0.30,   # m
}

z_c_num = z_c.subs(vrijednosti)
I_y_num = I_y.subs(vrijednosti)

print(f"\nNumerička provjera (bf=40cm, tf=10cm, bw=15cm, hw=30cm):")
print(f"Težište z_c = {z_c_num*100:.1f} cm od dna")
print(f"I_y = {I_y_num*1e8:.1f} cm⁴")


---
## 6. Linearna algebra — Matrica krutosti grednog elementa

### Ključne naredbe

| Naredba | Opis |
|---|---|
| `Matrix([[...],[...]])` | Kreiranje matrice |
| `A * B` | Množenje matrica |
| `A.T` | Transponirana matrica |
| `A.inv()` | Inverz matrice |
| `A.det()` | Determinanta |
| `A.eigenvals()` | Svojstvene vrijednosti |

### Matrica krutosti grednog elementa (Euler-Bernoulli):

$$[K_e] = \frac{EI}{L^3} \begin{bmatrix}
12 & 6L & -12 & 6L \\
6L & 4L^2 & -6L & 2L^2 \\
-12 & -6L & 12 & -6L \\
6L & 2L^2 & -6L & 4L^2
\end{bmatrix}$$

DOF redoslijed: `[v₁, φ₁, v₂, φ₂]`

In [ ]:
# ── Zadane varijable ─────────────────────────────────────────────────────
E, I, L = symbols('E I L', positive=True)

# ── TODO: Matrica krutosti grednog elementa ───────────────────────────────
# K_e = (E*I/L**3) * Matrix([[12, 6*L, -12, 6*L],
#                             [6*L, 4*L**2, -6*L, 2*L**2],
#                             [-12, -6*L, 12, -6*L],
#                             [6*L, 2*L**2, -6*L, 4*L**2]])

K_e = ...   # TODO: upiši izraz

print("Matrica krutosti grednog elementa K_e:")
display(K_e)

In [ ]:
# ── Analiza matrice krutosti ──────────────────────────────────────────────
# Determinanta
det_Ke = K_e.det()
print("Determinanta det(K_e):", simplify(det_Ke))
# Determinanta je 0 → matrica je singularna (nestabilna bez oslonca)

# ── TODO: Kondenzirana matrica (uklanjamo DOF v1, phi1 — oslonac A uklješten) ─
# Ostaju DOF: v2, phi2  →  podmatrica 2x2 (redovi/stupci 2,3 = indeksi 2,3)
K_cond = K_e[2:4, 2:4]   # Python slice: redovi 2-3, stupci 2-3
print("Kondenzirana matrica krutosti (konzola - slobodni kraj):")
display(K_cond)

# Inverzija kondenzirane matrice = matrica fleksibilnosti
F_cond = K_cond.inv()
print("Matrica fleksibilnosti (inverzija matrice K_cond):")
display(simplify(F_cond))

In [ ]:
# ── TODO: Provjera — sila i progib ────────────────────────────────────────
# Na slobodnom kraju konzole (v2, phi2) djeluje sila F i moment M_tip
F_sila  = symbols('F', positive=True)
M_tip   = symbols('M_tip', positive=True)

# Vektor opterećenja {p} = {F, M_tip}
p_vec = Matrix([F_sila, M_tip])

# Pomaci = F_cond * p_vec  →  {v2, phi2}
pomaci = simplify(F_cond * p_vec)

print("Progib i rotacija slobodnog kraja konzole:")
print("v_2 (progib):")
display(pomaci[0])

print("θ_2 (rotacija):")
display(pomaci[1])

# Za čistu silu M_tip=0 trebamo dobiti: v2 = F*L³/(3EI), phi2 = F*L²/(2EI)
v2_provjera = pomaci[0].subs(M_tip, 0)

print("Progib konzole pod silom F (M=0):", simplify(v2_provjera))
print("Standardna formula F*L³/(3EI):", Rational(1,3)*F_sila*L**3/(E*I))

---
## 7. Kombinacije opterećenja prema EN 1990

### Eurokod kombinacije za granično stanje nosivosti (ULS)

**STR/GEO kombinacija (6.10a / 6.10b):**

$$E_d = \sum_j \gamma_{G,j} G_{k,j} + \gamma_Q Q_{k,1} + \sum_{i>1} \gamma_{Q,i} \psi_{0,i} Q_{k,i}$$

**Parcijalni faktori sigurnosti (tablica NA.A1.2(B)):**

| Djelovanje | Nepovoljna | Povoljna |
|---|---|---|
| Stalno G | γ_G = 1.35 | γ_G = 1.00 |
| Promjenljivo Q | γ_Q = 1.50 | γ_Q = 0.00 |

**Faktori kombinacije ψ₀ (tablica A1.1):**

| Kategorija | ψ₀ |
|---|---|
| A — stanovanje | 0.7 |
| B — uredi | 0.7 |
| Snijeg (H ≤ 1000m) | 0.5 |
| Vjetar | 0.6 |

In [ ]:
# ── Zadane varijable — Stambena zgrada (tlo + kat) ─────────────────────
# Karakteristična opterećenja [kN/m²]
gk1 = symbols('g_k1', positive=True)   # vlastita težina konstrukcije
gk2 = symbols('g_k2', positive=True)   # stalno nadometanje (estrih, pod)
qk1 = symbols('q_k1', positive=True)   # korisno opterećenje (kat. A)
qk2 = symbols('q_k2', positive=True)   # snijeg
qk3 = symbols('q_k3', positive=True)   # vjetar

# Parcijalni faktori (EN 1990, tablica A1.2(B))
gG_neg = Rational(135, 100)    # 1.35 — stalno nepovoljna strana
gG_pov = Rational(100, 100)    # 1.00 — stalno povoljna strana
gQ     = Rational(150, 100)    # 1.50 — promjenljivo

# Faktori kombinacije psi_0
psi0_A    = Rational(7, 10)    # 0.70 — korisno kat. A (stanovanje)
psi0_snij = Rational(5, 10)    # 0.50 — snijeg
psi0_vjet = Rational(6, 10)    # 0.60 — vjetar

print("Parcijalni faktori i faktori kombinacije definirani kao Rational → točni razlomci.")
print(f"gamma_G = {gG_neg}, gamma_Q = {gQ}")
print(f"psi_0 (kat.A) = {psi0_A}, psi_0 (snijeg) = {psi0_snij}, psi_0 (vjetar) = {psi0_vjet}")

In [ ]:
# ── TODO: ULS kombinacije (EN 1990 jednadžba 6.10) ─────────────────────
# Kombinacija 1: Q_k1 (korisno) je osnovno promjenljivo djelovanje
# Ed1 = gG_neg*(gk1+gk2) + gQ*qk1 + gQ*psi0_snij*qk2 + gQ*psi0_vjet*qk3

Ed1 = ...   # TODO: upiši izraz

# Kombinacija 2: Q_k2 (snijeg) je osnovno promjenljivo djelovanje
# Ed2 = gG_neg*(gk1+gk2) + gQ*psi0_A*qk1 + gQ*qk2 + gQ*psi0_vjet*qk3

Ed2 = ...   # TODO: upiši izraz

# Kombinacija 3: Q_k3 (vjetar) je osnovno promjenljivo djelovanje
# Ed3 = gG_neg*(gk1+gk2) + gQ*psi0_A*qk1 + gQ*psi0_snij*qk2 + gQ*qk3

Ed3 = ...   # TODO: upiši izraz

print("ULS kombinacija 1 (osnovno: korisno opterećenje):")
display(Eq(symbols('E_d1'), expand(Ed1)))

print("ULS kombinacija 2 (osnovno: snijeg):")
display(Eq(symbols('E_d2'), expand(Ed2)))

print("ULS kombinacija 3 (osnovno: vjetar):")
display(Eq(symbols('E_d3'), expand(Ed3)))

In [ ]:
# ── Mjerodavna (najveća) kombinacija ─────────────────────────────────────
# Za numeričku provjeru uvrstiti tipične vrijednosti [kN/m²]
num_vrijednosti = {
    gk1: 4.0,   # vlastita težina konstrukcije [kN/m²]
    gk2: 1.5,   # stalno nadometanje [kN/m²]
    qk1: 2.0,   # korisno opterećenje kat. A [kN/m²]
    qk2: 0.8,   # snijeg [kN/m²]
    qk3: 0.6    # vjetar [kN/m²]
}

# ── TODO: Numeričko uvrštavanje ───────────────────────────────────────────
# Koristimo .subs(num_vrijednosti).evalf()
Ed1_num = ...   # TODO Ed1.subs(num_vrijednosti).evalf()
Ed2_num = ...   # TODO Ed2.subs(num_vrijednosti).evalf()
Ed3_num = ...   # TODO Ed3.subs(num_vrijednosti).evalf()

print(f"E_d1 = {float(Ed1_num):.3f} kN/m²")
print(f"E_d2 = {float(Ed2_num):.3f} kN/m²")
print(f"E_d3 = {float(Ed3_num):.3f} kN/m²")

Ed_max = max(float(Ed1_num), float(Ed2_num), float(Ed3_num))
print(f"\nMjerodavna ULS kombinacija: E_d,max = {Ed_max:.3f} kN/m²")

---
## 8. Potresno inženjerstvo — SDOF sustav i elastični spektar (EC8)

### Sustav s jednim stupnjem slobode (SDOF)

$$m \ddot{u} + c \dot{u} + k u = -m \ddot{u}_g(t)$$

Vlastita frekvencija:
$$\omega_n = \sqrt{\frac{k}{m}}, \quad T_n = \frac{2\pi}{\omega_n}, \quad f_n = \frac{1}{T_n}$$

### Elastični spektar odgovora (EC8-1, §3.2.2)

Za tip tla B, seizmičku zonu `a_gR`, faktor važnosti `γ_I`:

$$a_g = \gamma_I \cdot a_{gR}$$

$$0 \leq T \leq T_B: \quad S_e(T) = a_g \cdot S \left[ 1 + \frac{T}{T_B}(\eta \cdot 2.5 - 1) \right]$$
$$T_B \leq T \leq T_C: \quad S_e(T) = a_g \cdot S \cdot \eta \cdot 2.5$$
$$T_C \leq T \leq T_D: \quad S_e(T) = a_g \cdot S \cdot \eta \cdot 2.5 \cdot \frac{T_C}{T}$$
$$T_D \leq T \leq 4s: \quad S_e(T) = a_g \cdot S \cdot \eta \cdot 2.5 \cdot \frac{T_C T_D}{T^2}$$

In [ ]:
# ── Zadane varijable — SDOF i spektar ────────────────────────────────────
m_mass, k_krutost = symbols('m k', positive=True)   # masa [kg], krutost [kN/m]
T, TB, TC, TD     = symbols('T T_B T_C T_D', positive=True)   # periodi [s]
ag, S_faktor      = symbols('a_g S', positive=True)  # projektno ubrzanje tla, faktor tla
eta               = symbols('eta', positive=True)    # faktor prigušenja (5%: eta=1.0)

# ── TODO: Vlastita frekvencija SDOF sustava ───────────────────────────────
# omega_n = sqrt(k_krutost/m_mass)
omega_n = ...   # TODO

# T_n = 2*pi / omega_n
T_n     = ...   # TODO

# f_n = 1 / T_n
f_n     = ...   # TODO

print("Kružna vlastita frekvencija ω_n:")
display(Eq(symbols('omega_n'), omega_n))

print("Vlastiti period T_n:")
display(Eq(symbols('T_n'), T_n))

print("Vlastita frekvencija f_n:")
display(Eq(symbols('f_n'), f_n))

In [ ]:
# ── TODO: Elastični spektar EC8 — grana T_C leq T \leq T_D ─────────────
# Se(T) = ag * S_faktor * eta * 2.5 * (TC/T)     za TC <= T <= TD

Se_CD = ...   # TODO: upiši izraz

print("Spektralno ubrzanje (grana TC-TD):")
display(Eq(symbols('S_e(T)'), Se_CD))

# ── Numerički spektar za tlo tip B u Hrvatskoj ────────────────────────────
# Tlo tip B: S=1.2, TB=0.15s, TC=0.5s, TD=2.0s
# Seizmička zona: agR = 0.1g = 0.981 m/s², gamma_I = 1.0 (kateg. II)
import numpy as np
import matplotlib.pyplot as plt

g_grav = 9.81
agR    = 0.10 * g_grav   # m/s²
gI     = 1.00
ag_num = gI * agR

S_num  = 1.20
TB_num = 0.15
TC_num = 0.50
TD_num = 2.00
eta_num = 1.00   # 5% prigušenja

# ── TODO: Spektralna ubrzanja po granama ─────────────────────────────────
T_arr = np.linspace(0.01, 4.0, 500)
Se_arr = np.zeros_like(T_arr)

for i, T_val in enumerate(T_arr):
    if T_val <= TB_num:
        # Grana 0 → TB: Se = ag*S*(1 + T/TB*(eta*2.5-1))
        Se_arr[i] = ag_num * S_num * (1 + T_val/TB_num * (eta_num*2.5 - 1))
    elif T_val <= TC_num:
        # Grana TB → TC: Se = ag*S*eta*2.5
        Se_arr[i] = ag_num * S_num * eta_num * 2.5
    elif T_val <= TD_num:
        # Grana TC → TD: Se = ag*S*eta*2.5*(TC/T)
        Se_arr[i] = ag_num * S_num * eta_num * 2.5 * (TC_num / T_val)
    else:
        # Grana TD → 4s: Se = ag*S*eta*2.5*(TC*TD/T²)
        Se_arr[i] = ag_num * S_num * eta_num * 2.5 * (TC_num * TD_num / T_val**2)

plt.figure(figsize=(9, 5))
plt.plot(T_arr, Se_arr, 'b-', linewidth=2, label='Elastični spektar (EC8, tlo B)')
plt.axvline(TB_num, color='gray', linestyle='--', alpha=0.6, label=f'$T_B$ = {TB_num}s')
plt.axvline(TC_num, color='orange', linestyle='--', alpha=0.6, label=f'$T_C$ = {TC_num}s')
plt.axvline(TD_num, color='red', linestyle='--', alpha=0.6, label=f'$T_D$ = {TD_num}s')
plt.xlabel('Period T [s]', fontsize=12)
plt.ylabel('$S_e(T)$ [m/s²]', fontsize=12)
plt.title(f'Elastični spektar odgovora — EC8, tlo tip B, $a_{{gR}}$ = 0.10g', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── TODO: Potresna sila na SDOF zgradu ───────────────────────────────────
# Baza: m = 500 kN / g = masa (masa iz težine G_k)
# k = 20 000 kN/m → krutost konstrukcije

Gk_zgrada  = 5000.0   # [kN] — ukupna težina zgrade
k_zgrada   = 20000.0  # [kN/m] — lateralna krutost zgrade

m_kg  = Gk_zgrada / g_grav * 1000   # pretvorba kN → N → kg
T_val = 2 * np.pi * np.sqrt(m_kg / (k_zgrada * 1000))   # [s]

# Spektralno ubrzanje za T_val
if T_val <= TB_num:
    Se_val = ag_num * S_num * (1 + T_val/TB_num * (eta_num*2.5 - 1))
elif T_val <= TC_num:
    Se_val = ag_num * S_num * eta_num * 2.5
elif T_val <= TD_num:
    Se_val = ag_num * S_num * eta_num * 2.5 * TC_num / T_val
else:
    Se_val = ag_num * S_num * eta_num * 2.5 * TC_num * TD_num / T_val**2

# Potresna sila: F_b = Se * m  [N] → [kN]
Fb = Se_val * m_kg / 1000   # [kN]

print(f"Vlastiti period zgrade: T_n = {T_val:.3f} s")
print(f"Spektralno ubrzanje:    S_e = {Se_val:.4f} m/s²")
print(f"Potresna baza sila:     F_b = {Fb:.1f} kN")
print(f"Udio potresne sile:     F_b/G_k = {Fb/Gk_zgrada:.3f} = {Fb/Gk_zgrada*100:.1f}%")

---
## 9. Eulerovo izvijanje drvenih stupova (EC5)

### Drveni stup — Eulerova kritična sila

$$N_{cr} = \frac{\pi^2 E I}{(L_{eff})^2} = \frac{\pi^2 E I}{(k \cdot L)^2}$$

Faktor duljine izvijanja `k`: `k=1.0` (obostrano zglobno), `k=0.5` (obostrano uklješteno),  
`k=0.7` (dolje uklješteno, gore slobodno klizanje), `k=2.0` (konzola)

In [ ]:
from sympy import symbols, sqrt, pi, Rational, Eq, simplify

# ── Zadane varijable — Drveni stup ───────────────────────────────────────
E_d, I_d    = symbols('E I', positive=True)    # modul elastičnosti i tromosti [N/mm², mm⁴]
L_stup, k_s = symbols('L k', positive=True)    # duljina stupa, faktor duljine izvijanja [-]
A_s         = symbols('A', positive=True)      # površina presjeka [mm²]
fc0k        = symbols('f_c0k', positive=True)  # karakteristična tlačna čvrstoća [MPa]

# Parcijalni faktori (Rational zbog točnosti simboličkog računa)
gamma_M5    = Rational(130, 100)               # gamma_M5 = 1.30 (EC5, drvo C24)
k_mod       = Rational(80, 100)                # k_mod = 0.80 (klasa korištenja 1, stalno)

# ── TODO: Eulerova sila i vitost ─────────────────────────────────────────
# N_cr = pi**2 * E_d * I_d / (k_s * L_stup)**2
N_cr = ...   # TODO

# Polumjer tromosti: i = sqrt(I/A)
i_rad = sqrt(I_d / A_s)

# Vitost: lambda = k*L / i
lambda_rel = k_s * L_stup / i_rad
lambda_rel = simplify(lambda_rel)

# Projektna tlačna čvrstoća drva: fc0d = kmod * fc0k / gamma_M5
fc0d = k_mod * fc0k / gamma_M5

print("Eulerova kritična sila:")
display(Eq(symbols('N_cr'), N_cr))

print("Vitost stupa λ:")
display(Eq(symbols('lambda'), lambda_rel))

print("Projektna tlačna čvrstoća (EC5):")
display(Eq(symbols('f_c0d'), fc0d))

In [ ]:
# ── Numerička provjera — stup 120x120 mm, L=3.0m, drvo C24 ─────────────
# C24: E_0,mean = 11 000 MPa, f_c0k = 21 MPa

vrijednosti_drvo = {
    E_d:   11000,        # MPa
    I_d:   120**4 / 12,  # mm⁴ (kvadratni presjek 120x120)
    A_s:   120 * 120,    # mm²
    L_stup: 3000,        # mm
    k_s:   1.0,          # obostrano zglobno
    fc0k:  21.0          # MPa
}

N_cr_num     = N_cr.subs(vrijednosti_drvo) / 1000     # kN
lambda_num   = lambda_rel.subs(vrijednosti_drvo)
fc0d_num     = fc0d.subs(vrijednosti_drvo)

print(f"Eulerova kritična sila: N_cr = {N_cr_num:.1f} kN")
print(f"Vitost stupa: λ = {lambda_num:.1f}")
print(f"Projektna tlačna čvrstoća: f_c0d = {fc0d_num:.2f} MPa")
print(f"Projektna tlačna nosivost (bez izvijanja): N_Rd = {fc0d_num * 120*120 / 1000:.1f} kN")

---
## 📝 Sažetak — SymPy naredbe korištene u ovom tjednu

| Naredba | Opis | Primjer iz nastave |
|---|---|---|
| `symbols(...)` | Deklaracija simboličkih varijabli | geometrija presjeka, opterećenja |
| `Eq(l, d)` | Jednadžba l = d | uvjeti ravnoteže |
| `solve(j, x)` | Rješavanje jednadžbe/sustava | reakcije oslonaca |
| `diff(f, x, n)` | n-ta derivacija | elastična linija |
| `integrate(f, (x,a,b))` | Određeni integral | moment tromosti |
| `expand/factor/simplify` | Algebarska manipulacija | Wpl, N_cr |
| `Matrix(...)` | Matrica | matrica krutosti |
| `.subs(dict)` | Supstitucija vrijednosti | numeričke provjere |
| `.evalf()` | Numerička evaluacija | decimalni rezultati |
| `display(izraz)` | LaTeX render u Colabu | sve sekcije |
| `Rational(p, q)` | Točan razlomak p/q | Eurokod faktori |
